# Kernel SHAP for Sentiment140: Explaining Twitter Sentiment Predictions

**Project component:** Kernel SHAP  
**Team:** Kumar Kartikeyean (22MA3AI28), Nikhil Gupta, Raghav  
**Dataset:** [Sentiment140 — Kaggle (kazanova/sentiment140)](https://www.kaggle.com/datasets/kazanova/sentiment140)  
**Reference implementation:** [SHAP GitHub](https://github.com/shap/shap) · [KernelExplainer documentation](https://shap.readthedocs.io/en/stable/generated/shap.KernelExplainer.html)

## Objective

1. Train and evaluate a sentiment classifier on **real tweets from Sentiment140**.
2. Apply **`shap.KernelExplainer`**, not Tree SHAP, to explain predicted **positive-sentiment probabilities**.
3. Present individual explanations (waterfall plots), aggregate feature importance (bar and beeswarm plots), and verify **local additivity**.
4. Discuss the method, limitations, and computational trade-offs.

**Important:** This notebook has no fabricated results. Execute the cells to populate metrics and figures using the dataset. The default settings sample the 1.6M-row dataset and explain a small group of tweets for reasonable runtime on a CPU.

## 1. Environment Setup and Running 

Run in **Kaggle**, **Google Colab**, or local Jupyter with Python 3.10+. In Kaggle, use **Add Input → Sentiment140**; in Colab/local, the notebook attempts a public `kagglehub` download if it cannot find the CSV. Alternatively, manually download the dataset and put `training.1600000.processed.noemoticon.csv` in a `data/` folder.

The package-install cell may be skipped if everything is already installed. Re-run the kernel after installation if your environment requests it. Adjust the parameters below to trade execution time against quality.

In [ ]:
%pip -q install "shap>=0.46,<0.52" "kagglehub>=0.3" "scikit-learn>=1.3" "pandas>=2.0" "matplotlib>=3.7" "seaborn>=0.12"

In [ ]:
import os
import re
import html
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import sklearn
from scipy import sparse
from IPython.display import display
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)

SEED = int(os.getenv("KERNEL_SHAP_SEED", "42"))
SAMPLE_FRACTION = float(os.getenv("KERNEL_SHAP_SAMPLE_FRAC", "0.08"))
MAX_PER_CLASS = int(os.getenv("KERNEL_SHAP_MAX_PER_CLASS", "50000"))
MAX_FEATURES = int(os.getenv("KERNEL_SHAP_MAX_FEATURES", "400"))
BACKGROUND_SIZE = int(os.getenv("KERNEL_SHAP_BACKGROUND_SIZE", "15"))
N_EXPLAIN = int(os.getenv("KERNEL_SHAP_N_EXPLAIN", "12"))
SHAP_NSAMPLES = int(os.getenv("KERNEL_SHAP_NSAMPLES", "400"))

np.random.seed(SEED)
sns.set_theme(style="whitegrid", context="notebook")
OUTPUT_DIR = Path("kernel_shap_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python packages: SHAP={shap.__version__}, scikit-learn={sklearn.__version__}")
print(f"Settings: max_features={MAX_FEATURES}, background={BACKGROUND_SIZE}, "
      f"tweets_to_explain={N_EXPLAIN}, nsamples={SHAP_NSAMPLES}")

## 2. Load Sentiment140

The original training file has **no header** and contains columns `target`, `id`, `date`, `query`, `user`, and `text`. The training labels are `0` (negative) and `4` (positive); we map them to `0` and `1`. Read with **Latin-1 encoding** to avoid common decoding errors.

**Why chunked sampling?** The file is ordered by sentiment in many releases. Taking the *first* N rows can inadvertently select just one class. Instead, we scan **all chunks** and randomly sample from each. We keep only the label and text fields for this task.

In [ ]:
FILENAME = "training.1600000.processed.noemoticon.csv"


def find_sentiment140_csv():
    # 1) Explicit local path (useful for shared labs / test environments).
    provided = os.getenv("SENTIMENT140_CSV")
    if provided:
        p = Path(provided).expanduser()
        if not p.is_file():
            raise FileNotFoundError(f"SENTIMENT140_CSV does not exist: {p}")
        return p

    # 2) Kaggle input and typical local folders.
    for base in (Path("/kaggle/input/sentiment140"), Path("/kaggle/input"),
                 Path("data"), Path(".")):
        if not base.exists():
            continue
        matches = list(base.rglob(FILENAME))
        if matches:
            return matches[0]

    # 3) Download the specified public Kaggle dataset.
    print("CSV not found locally. Attempting Kaggle download...")
    try:
        import kagglehub
        downloaded = Path(kagglehub.dataset_download("kazanova/sentiment140"))
        matches = list(downloaded.rglob(FILENAME))
        if matches:
            return matches[0]
    except Exception as e:
        print(f"Automatic download unavailable: {type(e).__name__}: {e}")

    raise FileNotFoundError(
        "Could not find Sentiment140. Download the Kaggle dataset and put "
        f"{FILENAME} in ./data/ or set SENTIMENT140_CSV to the full path."
    )


csv_path = find_sentiment140_csv()
print("Source dataset:", csv_path)

columns = ["target", "id", "date", "query", "user", "text"]
sampled_parts = []
observed_counts = {0: 0, 4: 0}

for part_idx, chunk in enumerate(pd.read_csv(
    csv_path, header=None, names=columns, usecols=["target", "text"],
    encoding="latin-1", chunksize=100_000, on_bad_lines="skip"
)):
    chunk = chunk[chunk["target"].isin([0, 4])].dropna(subset=["text"])
    counts = chunk["target"].value_counts()
    for cls in (0, 4):
        observed_counts[cls] += int(counts.get(cls, 0))
    # Random sample, NOT head(): works even if the source CSV is class-sorted.
    sampled_parts.append(chunk.sample(frac=SAMPLE_FRACTION, random_state=SEED + part_idx))

if not sampled_parts:
    raise ValueError("The CSV contained no valid tweet rows.")

data = pd.concat(sampled_parts, ignore_index=True)
# An upper bound per class keeps the training time and memory manageable.
data = pd.concat(
    [g.sample(n=min(len(g), MAX_PER_CLASS), random_state=SEED)
     for _, g in data.groupby("target")],
    ignore_index=True
)
data["label"] = (data["target"] == 4).astype("int8")

assert data["label"].nunique() == 2, "Both sentiment classes are needed."
print("Valid tweets in entire source (by original label):", observed_counts)
print(f"Tweets selected for this experiment: {len(data):,}")
display(data[["text", "label"]].sample(min(5, len(data)), random_state=SEED))

## 3. Explore and clean the tweets

We remove links and username mentions, retain hashtag words, unescape HTML entities, and lowercase the text. **Negations** such as `not`, `don't`, and `never` are retained because they can reverse sentiment. Punctuation, spelling, sarcasm, and word order remain limitations of a bag-of-words model.

In [ ]:
def clean_tweet(text):
    text = html.unescape(str(text)).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


data["clean_text"] = data["text"].map(clean_tweet)
data = data[data["clean_text"].str.len() > 2].drop_duplicates(subset=["clean_text"])
data = data.reset_index(drop=True)

print("Class distribution after cleaning and deduplication:")
display(data["label"].map({0: "Negative", 1: "Positive"}).value_counts().to_frame("tweets"))
ax = data["label"].map({0: "Negative", 1: "Positive"}).value_counts().plot.bar(
    figsize=(5.5, 3.2), color=["#C65A5A", "#4A92A6"], rot=0
)
ax.set(xlabel="Sentiment", ylabel="Number of tweets", title="Selected Sentiment140 sample")
plt.tight_layout()
plt.show()

## 4. Train/test split and text-to-feature mapping

We use **binary word-presence features**: each feature is `1` if a vocabulary word occurs in a tweet and `0` otherwise. This directly matches Kernel SHAP's idea of *including or masking a feature*, and yields readable feature names on plots. We deliberately keep the vocabulary modest (`MAX_FEATURES=400`) for Kernel SHAP tractability.

We fit the vectorizer **only on training data** to avoid information leakage. We train Logistic Regression as a simple, reproducible baseline. **Kernel SHAP is still genuinely model-agnostic:** it receives only the function that returns predicted probabilities; it does not inspect the logistic-regression coefficients.

In [ ]:
X_text_train, X_text_test, y_train, y_test = train_test_split(
    data["clean_text"], data["label"].astype(int),
    test_size=0.20, stratify=data["label"], random_state=SEED
)

vectorizer = CountVectorizer(
    binary=True, max_features=MAX_FEATURES, min_df=3,
    ngram_range=(1, 1),
    token_pattern=r"(?u)\b[a-z][a-z']+\b",  # preserve contractions such as "don't"
    lowercase=False  # already lowercased by clean_tweet()
)
X_train = vectorizer.fit_transform(X_text_train)
X_test = vectorizer.transform(X_text_test)
feature_names = vectorizer.get_feature_names_out().tolist()

model = LogisticRegression(max_iter=1000, solver="liblinear", C=2.0, random_state=SEED)
model.fit(X_train, y_train)

print(f"Train samples: {X_train.shape[0]:,}, test samples: {X_test.shape[0]:,}")
print(f"Vocabulary / number of model features: {len(feature_names):,}")
print("Feature examples:", feature_names[:20])

## 5. Evaluate predictive performance

We report hold-out **accuracy**, **ROC-AUC**, a **classification report**, and the **confusion matrix**. Prediction quality and explanation quality are different questions: SHAP describes *why the fitted model behaved as it did*, not whether the prediction is correct.

In [ ]:
prob_test = model.predict_proba(X_test)[:, 1]
pred_test = (prob_test >= 0.5).astype(int)

metrics = {
    "accuracy": accuracy_score(y_test, pred_test),
    "roc_auc": roc_auc_score(y_test, prob_test),
}
print("Hold-out metrics:", {k: round(v, 4) for k, v in metrics.items()})
print("\nClassification report:\n")
print(classification_report(y_test, pred_test, target_names=["Negative", "Positive"]))

ConfusionMatrixDisplay.from_predictions(
    y_test, pred_test, display_labels=["Negative", "Positive"], cmap="Blues"
)
plt.title("Sentiment classifier — held-out tweets")
plt.tight_layout()
plt.show()

## 6. Kernel SHAP: mathematical foundation

For a tweet represented by $x\in\{0,1\}^M$, let $f(x)=P(\text{positive}\mid x)$ be our *black-box predictor*. Given a subset of included features $S\subseteq\{1,\dots,M\}$, Kernel SHAP evaluates perturbed samples by retaining the features in $S$ and replacing the other feature values with values from a **background distribution** $B$:

$$v_x(S) = \mathbb{E}_{b\sim B}\left[f(x_S,b_{\bar S})\right].$$

It then fits an additive, locally faithful surrogate

$$g(z') = \phi_0 + \sum_{j=1}^{M}\phi_j z'_j,\qquad z'_j\in\{0,1\}.$$

The subset samples in this weighted linear regression are weighted (for $0<|S|<M$) according to the **Shapley kernel**

$$\pi(S)=\frac{M-1}{\binom{M}{|S|}\,|S|\,(M-|S|)}.$$

The coefficients estimate Shapley feature attributions: positive $\phi_j$ **increases** the predicted positive probability relative to background; negative $\phi_j$ **decreases** it. With identity link, local additivity means

$$\boxed{f(x)\;\approx\;\mathbb{E}_{b\sim B}[f(b)] + \sum_j \phi_j.}$$

We use a **small background sample** to approximate missing-feature marginalization, and a restricted number of coalition evaluations (`SHAP_NSAMPLES`) to make model-agnostic explanation feasible. Kernel SHAP describes the *model and background choice*, not a causal effect of individual words.

## 7. Construct the black-box explanation function

The SHAP explainer sees only a callable `predict_positive(features)` and a representative **training-only background**. It does not receive the sentiment labels or the internal model coefficients. **Positive class** is column 1 in `predict_proba`.

Dense conversion is limited to the small background/explanation subsets; the large training and test matrices remain sparse.

In [ ]:
def predict_positive(feature_matrix):
    """Return P(positive), with one output per row (avoids multi-output SHAP issues)."""
    if isinstance(feature_matrix, pd.DataFrame):
        feature_matrix = feature_matrix.to_numpy()
    if sparse.issparse(feature_matrix):
        matrix = feature_matrix.tocsr()
    else:
        matrix = sparse.csr_matrix(np.asarray(feature_matrix, dtype=np.float64))
    return model.predict_proba(matrix)[:, 1]


rng = np.random.default_rng(SEED)
bg_indices = rng.choice(X_train.shape[0], size=min(BACKGROUND_SIZE, X_train.shape[0]), replace=False)
X_background = pd.DataFrame(X_train[bg_indices].toarray(), columns=feature_names)

print(f"Background shape: {X_background.shape}")
print(f"Background average P(positive): {predict_positive(X_background).mean():.4f}")

## 8. Explain representative unseen tweets

Choose a mix of **correctly classified positives**, **correctly classified negatives**, and **misclassified** tweets when available. This prevents us from showing only convenient cherry-picked examples. Explain a small subset of the test set — not the full test set — due to Kernel SHAP computational cost.

`l1_reg="num_features(20)"` performs feature selection in the weighted regression; it is **an approximation**, not a statement that only 20 words can matter.

In [ ]:
correct_pos = np.flatnonzero((y_test.to_numpy() == 1) & (pred_test == 1))
correct_neg = np.flatnonzero((y_test.to_numpy() == 0) & (pred_test == 0))
incorrect = np.flatnonzero(pred_test != y_test.to_numpy())

selected = []
for group, count in ((correct_pos, 3), (correct_neg, 3), (incorrect, 3)):
    selected.extend(group[:count].tolist())

for idx in rng.permutation(len(y_test)):
    if len(selected) >= min(N_EXPLAIN, len(y_test)):
        break
    if int(idx) not in selected:
        selected.append(int(idx))
selected = selected[:N_EXPLAIN]

X_explain = pd.DataFrame(X_test[selected].toarray(), columns=feature_names)
meta = pd.DataFrame({
    "tweet": X_text_test.iloc[selected].tolist(),
    "actual": y_test.iloc[selected].to_numpy(),
    "predicted": pred_test[selected],
    "p_positive": prob_test[selected]
}).reset_index(drop=True)

display(meta.assign(
    actual=meta["actual"].map({0: "Negative", 1: "Positive"}),
    predicted=meta["predicted"].map({0: "Negative", 1: "Positive"}),
    p_positive=meta["p_positive"].round(3)
))

start = time.perf_counter()
explainer = shap.KernelExplainer(
    predict_positive, X_background, feature_names=feature_names, link="identity"
)
raw_values = explainer.shap_values(
    X_explain,
    nsamples=SHAP_NSAMPLES,
    l1_reg="num_features(20)",
    silent=True
)
shap_values = np.asarray(raw_values, dtype=float)
if shap_values.ndim != 2:
    raise RuntimeError(f"Expected (tweets, features) SHAP matrix, got {shap_values.shape}")

base_value = float(explainer.expected_value)
explanations = shap.Explanation(
    values=shap_values,
    base_values=np.full(len(X_explain), base_value, dtype=float),
    data=X_explain.to_numpy(),
    feature_names=feature_names,
)
print(f"Kernel SHAP completed in {time.perf_counter() - start:.1f} seconds.")
print("SHAP matrix (tweets × features):", shap_values.shape)
print("Base P(positive):", round(base_value, 5))

## 9. Check local accuracy (the SHAP additivity property)

The reconstructed probability $\phi_0+\sum_j\phi_j$ should be close to the model probability for every explained tweet. This is a **numerical consistency test** of the explanation, not a test of whether the predicted sentiment is correct.

In [ ]:
model_probs = predict_positive(X_explain)
reconstructed_probs = base_value + shap_values.sum(axis=1)
additivity_error = model_probs - reconstructed_probs
additivity_table = pd.DataFrame({
    "model_P_positive": model_probs,
    "base_plus_SHAP_sum": reconstructed_probs,
    "absolute_error": np.abs(additivity_error),
    "actual_label": meta["actual"].to_numpy()
})
display(additivity_table.round(6))
print(f"Maximum absolute additivity error: {np.abs(additivity_error).max():.3g}")
assert np.allclose(model_probs, reconstructed_probs, atol=0.05), (
    "Additivity error too large: increase SHAP_NSAMPLES or inspect model/explainer configuration."
)

## 10. Local explanations: waterfall plots

Each plot starts at the background prediction $E[f(B)]$ and shows which vocabulary features move the prediction toward the final positive-class probability. For a **binary** feature, `0` indicates absence and `1` presence: **even absent words can have nonzero SHAP attributions** because the background sometimes contains them.

In [ ]:
for i in range(min(2, len(X_explain))):
    print(f"\nTweet {i+1}: {meta.loc[i, 'tweet']}")
    print(f"Actual: {'Positive' if meta.loc[i, 'actual'] else 'Negative'} | "
          f"Predicted P(positive): {model_probs[i]:.3f}")
    shap.plots.waterfall(explanations[i], max_display=12, show=False)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"waterfall_tweet_{i+1}.png", dpi=160, bbox_inches="tight")
    plt.show()

In [ ]:
# A readable contribution table for the first tweet, including whether a term is present.
i = 0
contrib_table = pd.DataFrame({
    "word": feature_names,
    "present_in_tweet": X_explain.iloc[i].astype(bool).to_numpy(),
    "SHAP_to_P_positive": shap_values[i]
})
contrib_table = contrib_table.reindex(
    contrib_table["SHAP_to_P_positive"].abs().sort_values(ascending=False).index
)
display(contrib_table.head(15).round(5))

## 11. Aggregate SHAP analysis across the explained tweets

Feature importance is approximated by **mean absolute SHAP value** across the selected explained tweets, not by model coefficients:

$$I_j=\frac{1}{n}\sum_{i=1}^{n}|\phi_j(x_i)|.$$

A **bar plot** ranks the average magnitude, while a **beeswarm plot** shows the range and sign of contributions across tweets. **Caution:** `N_EXPLAIN=12` is a small, purposely mixed subset, so this is illustrative *sample-level* aggregation, not a statistically robust estimate of global importance across 1.6M tweets.

In [ ]:
importance = pd.DataFrame({
    "word": feature_names,
    "mean_abs_SHAP": np.abs(shap_values).mean(axis=0),
    "mean_signed_SHAP": shap_values.mean(axis=0)
}).sort_values("mean_abs_SHAP", ascending=False)

display(importance.head(20).round(5))
importance.to_csv(OUTPUT_DIR / "kernel_shap_feature_importance.csv", index=False)

shap.plots.bar(explanations, max_display=15, show=False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "kernel_shap_bar.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
shap.plots.beeswarm(explanations, max_display=15, show=False)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "kernel_shap_beeswarm.png", dpi=160, bbox_inches="tight")
plt.show()

## 12. Compare positive/negative word contributions

For an **individual tweet**, the SHAP value's sign tells us whether the feature pushes the predicted **positive probability** upward or downward. These are explanations of the **probability** returned by the model, not the probability that a word is inherently positive or negative.

In [ ]:
for i in range(min(2, len(X_explain))):
    ranked = pd.DataFrame({
        "feature": feature_names,
        "value": X_explain.iloc[i].to_numpy(),
        "shap": shap_values[i]
    })
    positive_drivers = ranked.sort_values("shap", ascending=False).head(8)
    negative_drivers = ranked.sort_values("shap").head(8)
    print(f"\nTweet {i+1}: {meta.loc[i, 'tweet'][:130]}")
    print("Largest pushes toward POSITIVE (includes absent features):")
    display(positive_drivers.round(4))
    print("Largest pushes toward NEGATIVE (includes absent features):")
    display(negative_drivers.round(4))

## 13. Explain new custom tweets (optional)

The following cell demonstrates using the **same trained model and Kernel SHAP background** on hand-written tweets. These examples are *not* part of the evaluation set. Skip this cell if you only need test-set explanations.

In [ ]:
custom_tweets = [
    "I absolutely love this wonderful day!",
    "I hate this awful experience, it is terrible."
]
custom_clean = [clean_tweet(t) for t in custom_tweets]
X_custom = pd.DataFrame(vectorizer.transform(custom_clean).toarray(), columns=feature_names)
custom_probs = predict_positive(X_custom)

print(pd.DataFrame({"tweet": custom_tweets, "P_positive": custom_probs}).round(4).to_string(index=False))
custom_values = np.asarray(explainer.shap_values(
    X_custom, nsamples=SHAP_NSAMPLES, l1_reg="num_features(20)", silent=True
))
custom_exp = shap.Explanation(
    values=custom_values,
    base_values=np.full(len(X_custom), base_value),
    data=X_custom.to_numpy(),
    feature_names=feature_names
)
for i, label in enumerate(("positive", "negative")):
    shap.plots.waterfall(custom_exp[i], max_display=12, show=False)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"waterfall_custom_{label}.png", dpi=160, bbox_inches="tight")
    plt.show()

## 14. Save artifacts and reproducibility details

Export model, vectorizer, configuration, explained predictions, SHAP values, and figures. **Note:** saving the model uses `joblib` (a Python pickle-based format); only load it from trusted sources.

In [ ]:
import json
import joblib

joblib.dump(model, OUTPUT_DIR / "sentiment_logistic_regression.joblib")
joblib.dump(vectorizer, OUTPUT_DIR / "count_vectorizer.joblib")
np.save(OUTPUT_DIR / "shap_values.npy", shap_values)
meta.assign(base_probability=base_value,
            reconstructed_probability=reconstructed_probs,
            additivity_error=additivity_error).to_csv(
    OUTPUT_DIR / "explained_tweets.csv", index=False
)
config = {
    "source_dataset": "https://www.kaggle.com/datasets/kazanova/sentiment140",
    "seed": SEED,
    "fraction_read_per_chunk": SAMPLE_FRACTION,
    "max_per_class": MAX_PER_CLASS,
    "vocabulary_size": len(feature_names),
    "background_size": len(X_background),
    "explained_tweets": len(X_explain),
    "shap_coalition_samples": SHAP_NSAMPLES,
    "base_probability": base_value,
    "accuracy": float(metrics["accuracy"]),
    "roc_auc": float(metrics["roc_auc"]),
    "shap_version": shap.__version__,
    "sklearn_version": sklearn.__version__,
}
(OUTPUT_DIR / "experiment_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Files saved to:", OUTPUT_DIR.resolve())
print("Output files:", [p.name for p in sorted(OUTPUT_DIR.iterdir())])

## 15. Findings, interpretation and limitations (for presentation/report)

**How to interpret the executed outputs:**

- **Model validity:** Quote the actual hold-out accuracy and AUC printed in Section 5. The classifier is a baseline, not the best possible sentiment model.
- **Local explanations:** In waterfall plots, red/positive SHAP contributions shift $P(\mathrm{positive})$ up and blue/negative shift it down; the base value plus all contributions approximates the model prediction (verified in Section 9).
- **Aggregate importance:** Section 11 ranks influential word-presence features *within the small explained subset*. Re-run with a larger, randomly selected explanation sample for stronger global claims.
- **Model-agnostic nature:** `KernelExplainer` works through `predict_positive` and would also work with other classifiers accepting the same feature format; specialized explainers for linear or tree models are generally faster.

**Limitations and methodological cautions:**

1. **Background dependence:** SHAP attributions depend on which training tweets are used as the background. A background of 15 tweets is approximate; sensitivity tests with 30–50 are worthwhile if compute allows.
2. **Correlation and unrealistic coalitions:** Mixing word-presence features across tweets may create combinations that are linguistically unnatural. This is an inherent limitation of tabular-style feature masking on text.
3. **Bag-of-words:** A limited unigram vocabulary ignores much of word order, context, slang and sarcasm. Retaining negation words helps but does not solve this.
4. **Sampling and uncertainty:** We train on a random subset of the dataset, explain a tiny set of tweets, and approximate Shapley values with a finite number of coalitions. Results vary with data, background, and sampling seeds.
5. **Non-causality:** SHAP explains the fitted model's predictions; it does not prove that a word *caused* sentiment or that the dataset labels are error-free. Sentiment140 labels were derived using emoticons as noisy supervision.
6. **Class interpretation:** Every attribution here is for **class 1 = positive** and is in **probability units** (`link="identity"`), not log-odds.

**Suggested team presentation:** Introduce Kernel SHAP equations (Section 6), show evaluation metrics (Section 5), present two local waterfall plots (Section 10), show feature importance (Section 11), and discuss additivity/limitations (Sections 9 and 15).

### References

- [Sentiment140 dataset on Kaggle](https://www.kaggle.com/datasets/kazanova/sentiment140)
- [Go, Bhayani & Huang (2009), *Twitter Sentiment Classification using Distant Supervision*](https://cs.stanford.edu/people/alecmgo/papers/TwitterDistantSupervision09.pdf)
- [Lundberg & Lee (2017), *A Unified Approach to Interpreting Model Predictions*](https://arxiv.org/abs/1705.07874)
- [Official SHAP project](https://github.com/shap/shap)
- [Official KernelExplainer API](https://shap.readthedocs.io/en/stable/generated/shap.KernelExplainer.html)